# Prepare selected bike inputs

Review assigned to Viviana Pajic; review pending. Prepared with AI assistance. See [AI use](../docs/AI_USE.md).

## Keep time in order

Use only earlier counts to predict a day. Last week and recent averages carry useful patterns. Older dates train the model; later dates check it.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/review_models.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the project.')
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'mushrooms'), str(ROOT / 'citibike'), str(ROOT)]
import pandas as pd
from IPython.display import display
from weather_search import load_weather_features
from expanded_search import load_extended_weather_features
X_cpu, y_cpu, _ = load_weather_features()
X_gru, y_gru, _ = load_extended_weather_features()
display(pd.DataFrame([{'model': 'CPU weather', 'training_rows': len(X_cpu), 'last_target': str(X_cpu.index[-1].date())},
                      {'model': 'Expanded GRU weather', 'training_rows': len(X_gru), 'last_target': str(X_gru.index[-1].date())}]))
assert X_cpu.index.max() == X_gru.index.max() == pd.Timestamp('2023-06-30')

,model,training_rows,last_target
0,CPU weather,153,2023-06-30
1,Expanded GRU weather,518,2023-06-30


## Check missing prepared inputs

The first 28 days supply history. We do not fill absent weather with guesses. Scalers learn inside each model's training fold. This preparation notebook contains no graphs.

In [2]:
assert X_cpu.notna().all().all()
assert X_gru.notna().all().all()
print('Prepared training inputs have no missing values.')
print('Counts use preceding days. Calendar information is known in advance.')
print('No October-December targets were opened.')

Prepared training inputs have no missing values.
Counts use preceding days. Calendar information is known in advance.
No October-December targets were opened.


## Decision

CPU weather uses changes in forecasts, so it needs the target date plus seven earlier forecast dates. GRU weather needs only six forecast values for the target date. Both need 28 completed daily totals.

In [3]:
display(X_cpu.filter(regex='lag_|forecast_').head(2))

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,forecast_jma_day2_wind_max,forecast_temperature_range,forecast_temperature_change_1,forecast_temperature_change_7,forecast_temperature_weekend,forecast_warm_degree,forecast_cold_degree,forecast_rain_log,forecast_rain_change_7,forecast_wind_change_7
date,,,,,,,,,,,,,,,,,,,,,
2023-01-29,66819.0,65614.0,64439.0,41868.0,67647.0,38417.0,39438.0,58192.0,67532.0,32358.0,...,22.4,5.2,1.891667,2.354167,6.4,0.0,3.600000,0.000000,-7.8,7.216666
2023-01-30,58041.0,66819.0,65614.0,64439.0,41868.0,67647.0,38417.0,39438.0,58192.0,67532.0,...,13.5,5.5,1.345833,4.333333,0.0,0.0,2.254167,0.405465,-3.7,-11.629166
